In [1]:
import json
import os
import random
import shutil
from pathlib import Path
import torch

In [2]:
with open(r"D:\hyper-kvasir-segmented-images-extracted\segmented-images\bounding-boxes.json") as f:
    data = json.load(f)

In [9]:
len(data)

1000

In [10]:
os.makedirs('labels', exist_ok = True)

In [12]:
#create labels file with a single txt file for each image that contains
#center x, center y, width and height normalized for the image dimensions

for image_id, info in data.items():
    img_w = info['width']
    img_h = info['height']

    lines = []

    for bbox in info['bbox']:
        cx = (bbox['xmin'] + bbox['xmax'])/ 2 / img_w
        cy = (bbox['ymin'] + bbox['ymax']) / 2 / img_h
        w = (bbox['xmax'] - bbox['xmin']) / img_w
        h = (bbox['ymax'] - bbox['ymin']) / img_h
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    with open(f'labels/{image_id}.txt', 'w') as f:
        f.write('\n'.join(lines))

print(f"converted {len(data)} annotations")
                
            
              
        

converted 1000 annotations


In [14]:
images_src = r"D:\hyper-kvasir-segmented-images-extracted\segmented-images\images"
labels_src = r"D:\hyper-kvasir-segmented-images-extracted\segmented-images\labels"
output = r"D:\hyper-kvasir-segmented-images-extracted\polyp_detection"

In [17]:
for split in ['train', 'val']:
    os.makedirs(f"{output}/images/{split}", exist_ok = True)
    os.makedirs(f"{output}/labels/{split}", exist_ok = True)

image_ids = [f.stem for f in Path(labels_src).iterdir() if f.suffix == '.txt']



In [18]:
random.shuffle(image_ids)
split = int(0.8*len(image_ids))
train_ids = image_ids[:split]
val_ids = image_ids[split:]

In [19]:
for ids, folder in [(train_ids, 'train'), (val_ids, 'val')]:
    for img_id in ids:
        shutil.copy(f"{images_src}/{img_id}.jpg", f"{output}/images/{folder}/{img_id}.jpg")
        shutil.copy(f"{labels_src}/{img_id}.txt", f"{output}/labels/{folder}/{img_id}.txt")

print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")

Train: 800 | Val: 200


In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data = "polyp.yaml",
    epochs = 50,
    imgsz = 640,
    batch = 32,
    device = 0,

)


Ultralytics 8.4.184  Python-3.13.4 torch-2.14.1+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16284MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=polyp.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer

In [4]:
#inference on one image
model = YOLO('runs/detect/train/weights/best.pt')
results = model('data/0d040bb7-1973-494e-8d79-a462eda6f82e.jpg')
results[0].show()


image 1/1 C:\Users\docto\Python\resnet_50_endoscopy\data\0d040bb7-1973-494e-8d79-a462eda6f82e.jpg: 544x640 1 polyp, 7.7ms
Speed: 3.8ms preprocess, 7.7ms inference, 5.4ms postprocess per image at shape (1, 3, 544, 640)


In [7]:
model = YOLO('runs/detect/train/weights/best.pt')
video_results = model('data/polyp_recording.mp4', stream=True, save=True)
for result in video_results:
    pass


video 1/1 (frame 1/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 5.0ms
video 1/1 (frame 2/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 10.5ms
video 1/1 (frame 3/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 7.4ms
video 1/1 (frame 4/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 6.1ms
video 1/1 (frame 5/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 5.4ms
video 1/1 (frame 6/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 6.6ms
video 1/1 (frame 7/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 5.4ms
video 1/1 (frame 8/594) C:\Users\docto\Python\resnet_50_endoscopy\data\polyp_recording.mp4: 448x640 (no detections), 5.5ms
video 1/1 (fra